In [1]:
import numpy as np
import math
import os

In [2]:
# Signal to noise ratio, in discrete and continuous forms
disc_SNR_dB = [-5, 0, 5, 10, 15, 20]
cont_SNR_dB = np.random.uniform(-5, 20)

In [3]:
# RF signals
def generate_bits(bit_num):
    bits = np.random.randint(0, 2, bit_num)
    return bits

In [4]:
# BPSK
def bpsk_mod(bits):
    """Converts raw signals into BPSK modulated symbols
    0 -> -1
    1 -> +1    """
    bpsk_symbols = 2 * bits - 1
    return bpsk_symbols

In [5]:
def qpsk_mod(bits):
    """Converts raw signals into QPSK modulated symbols
    Considers a set of 2 bits
    1/sqrt(2) is used to normalize such that the average symbol power = 1
    00 -> 1 + j / sqrt(2)
    01 -> -1 + j/ sqrt(2)
    11 -> -1 -j/ sqrt(2)
    10 ->  1 - j/ sqrt(2)    """

    if len(bits)%2==0:
        pairs = bits.reshape(-1,2)

        qpsk_symbols = np.empty(len(pairs),dtype = np.complex64)

        for i, (bit1,bit2) in enumerate(pairs):
            # bit 1 -> complex part
            if bit1 == 0:
                imag = 1
            else:
                imag = -1

            # bit 2 -> real part    
            if bit2 == 0:
                real = 1
            else:
                real = -1
            qpsk_symbols[i] = (real + (imag*1j))/np.sqrt(2)
        return qpsk_symbols
    else :
        return None

In [6]:
def snr_transform(snr_db):
    """Converts SNR from dB to a linear form"""
    snr_linear = 10 ** (snr_db / 10)
    return snr_linear

In [7]:
def rayleigh(signal):
    """ Applies a rayleigh fading channel to the signal
    
    received signal = h * transmitted signal

    1/sqrt(2) is used to normalize such that the average symbol power = 1
    the underlying dist. of the real and imaginary components are gaussian
    """

    sigma = 1/np.sqrt(2)
    
    fade_coefficient = (
        np.random.normal(0,sigma,len(signal)) # I copmponent
        + 1j * np.random.normal(0,sigma,len(signal)) # Q copmponent
    )
    
    faded_signal = fade_coefficient * signal
    
    return faded_signal,fade_coefficient

In [8]:
def awgn(signal,snr_db,modulation):
    """Computes and Adds the required AWGN noise to the required bits
    
    received signal = transmitted signal + n

    diff sigmas for qpsk and bpsk since qpsk has twice the bits trasmitted per signal
    BPSK:  1 bit/symbol -> Eb = 1
    QPSK:  2 bits/symbol -> Eb = 1/2
    where Eb is the energy per bit
    """

    snr_linear = snr_transform(snr_db)
    
    # computes the standard deviation of the guassian dist.
    if modulation == "BPSK":
        sigma = np.sqrt(1 / (2 * snr_linear))
        
    elif modulation == "QPSK":
        sigma = np.sqrt(1 / (4 * snr_linear))

    awgn_noise = (
        np.random.normal(0,sigma,len(signal)) # I copmponent
        + 1j * np.random.normal(0, sigma, len(signal)) # Q copmponent
    )
    
    received_signal = signal + awgn_noise
    
    return received_signal,awgn_noise

In [9]:
# Sample data
bit_num = 64
bits = generate_bits(bit_num)
symbols = qpsk_mod(bits)

snr_db = np.random.choice(disc_SNR_dB)
print("SNR (dB): ",snr_db)


#print(faded_signal)
modulations = ["BPSK", "QPSK"]
for modulation in modulations:
    print("\n=================================================")
    print("Modulation: ",modulation)

    snr_linear = snr_transform(snr_db)
    faded_signal,h = rayleigh(symbols)
    print("\nAfter Rayleigh: ")
    received_signal,noise = awgn(symbols,snr_db,modulation)
    print("\nAfter AWGN: ")
    print(received_signal)

    print("Mean AWGN Noise: ")
    print(np.mean(np.abs(noise)**2))

SNR (dB):  20

Modulation:  BPSK

After Rayleigh: 

After AWGN: 
[ 0.74020965-0.77525218j -0.74269437+0.79686135j  0.58661514-0.75167615j
  0.71637717+0.69512052j  0.69410736-0.65615504j  0.73845439+0.64904995j
 -0.63963676-0.67082745j -0.75042558+0.62592327j  0.70593417+0.85187961j
  0.72645009+0.68499045j -0.65533243+0.66734095j -0.77055561-0.74051944j
  0.68205813+0.67675598j -0.78930113+0.65541601j  0.79177153-0.58690673j
  0.69270858+0.73239904j -0.75858184-0.66812491j  0.76924674+0.70186299j
 -0.63764371-0.61489833j  0.69312781-0.72089202j -0.66092898+0.68467547j
 -0.89570214-0.6807554j  -0.66608805-0.76119592j -0.75243094-0.70968694j
 -0.81638785-0.65251972j -0.66265199+0.51571463j  0.65891679-0.64371036j
  0.70171974+0.56069057j  0.71671438+0.71732909j -0.72789033-0.78522154j
  0.73571134-0.7836064j  -0.7170761 +0.69884532j]
Mean AWGN Noise: 
0.008755070362338023

Modulation:  QPSK

After Rayleigh: 

After AWGN: 
[ 0.81441031-0.76452243j -0.66875438+0.71720777j  0.70237419-0.72

In [10]:
"""Dataset Generation Test"""

#modulation = "QPSK"
modulation = "BPSK"

samples = 10000
signal_length = 1024

save_dir = f"../data/{modulation.lower()}"
os.makedirs(save_dir, exist_ok=True)

split_config = {
    "train": 10000,
    "test": 5000,
    "validation": 2000 
}

for split, samples in split_config.items():
    save_dir = os.path.join("..","data",modulation.lower(),split)

    os.makedirs(save_dir, exist_ok=True)

    if modulation == "BPSK":
        clean_signals = np.empty((samples,signal_length), dtype=np.float32)
        
    elif modulation == "QPSK":
        clean_signals = np.empty((samples, signal_length // 2), dtype=np.complex64)

    for i in range(samples):
        
        bits = generate_bits(signal_length)
    
        if modulation == "BPSK":
            symbols = bpsk_mod(bits).astype(np.float32)
            
        elif modulation == "QPSK":
            symbols = qpsk_mod(bits).astype(np.complex64)

        clean_signals[i] = symbols
    
    save_path = os.path.join(save_dir, "clean.npy")
    np.save(save_path, clean_signals)